# Generation of Fake Images with Checkboxes

Checkboxes often appear in formal documents. Checkboxes are not text, and check marks can have different forms (e.g. 'v' or 'x'), often appearing in a handwritten manner. Therefore, if the documents are scanned, which is often the case with handwritten check marks, it is not easy to extract the information about whether these boxes are checked or not, unless a detection model is trained beforehand. 
This code was originally developed to train a object localization model, which is useful to detect a single object. And so, it generates training images, containing one single check box with a check mark or none. Some other structures, such as tables or lines, randomly show up to make the localization model insensitive to noise or surrounding structures.
In the below samples, the red-line boxes are bounding boxes for check marks, indicating that a checkbox is checked, or a check mark is placed on or near the checkbox. 
The function can be easily generalized to contain multiple checkboxes. 


Bomsoo Kim

03/29/2022

In [ ]:
import matplotlib.font_manager
import pandas as pd

def find_all_font_names():
    data = []

    # ref) https://matplotlib.org/stable/api/font_manager_api.html#matplotlib.font_manager.FontProperties
    styles = ['normal','italic','oblique']
    variants = ['normal','small-caps']
    weights = ['normal','ultralight','light','regular','book','medium','roman','semibold','demibold','demi','bold','heavy','extra bold','black']

    #----------------------------------------------------------
    font_names = matplotlib.font_manager.get_font_names()
    for i, font_name in enumerate(font_names):
        # print(f'[{i+1}/{len(font_names)}] {font_name}')
        for style in styles:
            for variant in variants:
                for weight in weights:
                    font = matplotlib.font_manager.FontProperties(family=font_name, style=style, variant=variant, weight=weight)
                    filepath = matplotlib.font_manager.findfont(font)

                    data.append({
                        'font_name':font_name,
                        'style':style,
                        'variant':variant,
                        'weight':weight,
                        'filepath':filepath,                    
                    })

    #----------------------------------------------------------
    filepaths = matplotlib.font_manager.findSystemFonts()
    for filepath in filepaths:
        font = matplotlib.font_manager.get_font(filepath)

        data.append({
            'font_name':font.family_name,
            'filepath':filepath,
        })

    #----------------------------------------------------------
    list_fonts = pd.DataFrame(data)
    list_fonts = list_fonts.drop_duplicates(subset='filepath', keep='first')
    # print(list_fonts['filepath'].tolist())
    # list_fonts.to_excel('advanced_texts.xlsx', index=False)

    return list_fonts

In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageDraw, ImageFont

def create_example(shape=(50, 200, 3), line_color_threshold=50, list_font_filepaths=['arial.ttf'], draw_bbox=False):
    def draw_box(draw, x0, y0, x1, y1, x2, y2, line_color=0, line_width=1):
        draw.rectangle((x0, y0, x1, y1), outline=line_color, width=line_width)

    def draw_circle(draw, x0, y0, x1, y1, x2, y2, line_color=0, line_width=1):
        draw.ellipse((x0, y0, x1, y1), outline=line_color, width=line_width)

    def draw_X(draw, x0, y0, x1, y1, x2, y2, line_color=0, line_width=1):
        draw.line((x0, y0, x1, y1), fill=line_color, width=line_width)
        draw.line((x1, y0, x0, y1), fill=line_color, width=line_width)

    def draw_V(draw, x0, y0, x1, y1, x2, y2, line_color=0, line_width=1):
        draw.line((x0, y0, x2, y2), fill=line_color, width=line_width)
        draw.line((x2, y2, x1, y0), fill=line_color, width=line_width)    

        
    img = Image.fromarray(np.full(shape, 255, 'uint8')) # blank image
    
    #--- check box -----------------------------------
    width, height = random.uniform(10, 30), random.uniform(10, 30)

#     x0, y0 = random.uniform(0, img.size[0] - width), random.uniform(0, img.size[1] - height)
    x0, y0 = random.uniform(0, img.size[0] - width), random.uniform(0, 0.1*(img.size[1] - height))

    xm, ym = x0 + width/2, y0 + height/2 # center point of check box
    
    x1, y1 = x0 + width, y0 + height
    x2, y2 = x0 + width/2, y0 + height

    #--- table cell box -------------------------
    width_t, height_t = width * random.uniform(8, 12), height * random.uniform(1.5, 2.5)
    xx0, yy0 = xm - width_t/2, ym - height_t/2
    xx1, yy1 = xm + width_t/2, ym + height_t/2
    
    #--- check symbol -------------------------
    width_c, height_c = random.uniform(10, 30), random.uniform(10, 30)
    dx, dy = random.uniform(-width/2, width/2), random.uniform(-height/2, height/2)
    rx0, ry0 = xm - width_c/2 + dx + random.uniform(-3,3), ym - height_c/2 + dy + random.uniform(-3,3)
    rx1, ry1 = xm + width_c/2 + dx + random.uniform(-3,3), ym + height_c/2 + dy + random.uniform(-3,3)
    rx2, ry2 = xm             + dx + random.uniform(-3,3), ym + height_c/2 + dy + random.uniform(-3,3)
    
    
    draw = ImageDraw.Draw(img)

    #--- draw checkbox --------------------------
    is_checkbox = random.choice([0,1])
    if is_checkbox:
        line_color = (random.randrange(0, line_color_threshold), random.randrange(0, line_color_threshold), random.randrange(0, line_color_threshold))
        line_width = random.choice([1,2,3])
        
        draw_box(draw, x0, y0, x1, y1, x2, y2, line_color, line_width)

    #--- draw table cell box --------------------------
    is_table_cell = random.choice([0,1])
    if is_table_cell:
        line_color = (random.randrange(0, line_color_threshold), random.randrange(0, line_color_threshold), random.randrange(0, line_color_threshold))
        line_width = random.choice([1,2,3])
        
        draw.line((xx0, yy0, xx0, yy1), fill=line_color, width=line_width) # left vertical line
        draw.line((xx1, yy0, xx1, yy1), fill=line_color, width=line_width) # right vertical line
        if random.choice([0,1]):
            draw.line((0, yy0, xx1, yy0), fill=line_color, width=line_width) # upper horizontal line
        if random.choice([0,1]):
            draw.line((0, yy1, xx1, yy1), fill=line_color, width=line_width) # lower horizontal line
    
    #--- draw check symbol --------------------------
    is_checked = random.choice([0,1])
    check_symbol = random.choice(['X','V'])
    if is_checked:

        line_color = (random.randrange(0, line_color_threshold), random.randrange(0, line_color_threshold), random.randrange(0, line_color_threshold))
        line_width = random.choice([1,2,3])
        if check_symbol == 'X':
            draw_X(draw, rx0, ry0, rx1, ry1, rx2, ry2, line_color, line_width) # draw a check symbol
        elif check_symbol == 'V':
            draw_V(draw, rx0, ry0, rx1, ry1, rx2, ry2, line_color, line_width) # draw a check symbol
        else:
            raise(Exception(f'Brad error: no such check symbol: {check_symbol}...'))
    #--- draw text --------------------------
    text = '-------------------------------------------> )'
    font = ImageFont.truetype(random.choice(list_font_filepaths), int(height))

    # wt, ht = draw.textsize(text, font=font) # https://stackoverflow.com/questions/77038132/python-pillow-pil-doesnt-recognize-the-attribute-textsize-of-the-object-imag
    wt, ht = draw.textlength(text, font=font), int(height) # 2025-02-18
    
    if is_checkbox or is_table_cell or is_checked:
        draw.text((xx0 - wt - random.randrange(20,70), y0), text, font=font, align='right', fill='black')
    else:
        draw.text((shape[1] - wt - random.randrange(-20,shape[1]), y0), text, font=font, align='right', fill='black')
    
    #--- bounding box ----------------------------
#     bbox = (
#         max(min(x0,x1,x2,rx0,rx1,rx2)-1, -1), max(min(y0,y1,y2,ry0,ry1,ry2)-1, -1),
#         min(max(x0,x1,x2,rx0,rx1,rx2)+1, img.size[0]), min(max(y0,y1,y2,ry0,ry1,ry2)+1, img.size[1]),
#     ) if is_checked else (-1,-1,0,0)
    bbox = (
        (min(rx0,rx1), min(ry0,ry1), max(rx0,rx1), max(ry0,ry1))
        if check_symbol == 'X' else (
            (min(rx0,rx1,rx2), min(ry0,ry1,ry2), max(rx0,rx1,rx2), max(ry0,ry1,ry2))
            if check_symbol == 'V' else
            None
        )
    ) if is_checked else (-1,-1,0,0)
    bbox_norm = (bbox[0]/img.size[0], bbox[1]/img.size[1], bbox[2]/img.size[0], bbox[3]/img.size[1])
    
    
    img = np.array(img) # convert to numpy array
    
    #--- gauss randomize ---------------------------------
    random_gauss_interval = lambda mu, sig, th=255: int(abs(random.gauss(mu, sig)))%(th+1)
    
    iiib = (img < line_color_threshold) # black lines
    iiiw = (img == 255) # white background

#     sig_line = 50
#     sig_bkgn = 50
    sig_line = random_gauss_interval(0,50,51)
    sig_bkgn = random_gauss_interval(0,50,51)

    # img[iiib] = [j for j in [random_gauss_interval(0,sig_line) for i in range(len(img[iiib])//3)] for _ in range(3)] # same 3 channel
    img[iiib] = [j for j in [random_gauss_interval(0,sig_line) for i in range(len(img[iiib])//1)] for _ in range(1)] # random channel

#     img[iiiw] = [j for j in [255 - random_gauss_interval(0,sig_bkgn) for i in range(len(img[iiiw])//3)] for _ in range(3)] # same 3 channel
    img[iiiw] = [j for j in [255 - random_gauss_interval(0,sig_bkgn) for i in range(len(img[iiiw])//1)] for _ in range(1)] # random channel

    #--- black/white image ---------------------------------
    threshold_bw = 25
    if random.choice([0,1]):
        img[iiib[:,:,0] & (img[:,:,0] < threshold_bw)] = 0
        img[iiib[:,:,0] & (img[:,:,0] >= threshold_bw)] = 255
        img[iiiw[:,:,0] & (img[:,:,0] > threshold_bw)] = 255
        img[iiiw[:,:,0] & (img[:,:,0] <= threshold_bw)] = 0

    #--- draw bounding box for check -----------------------
    if draw_bbox: 
        img = Image.fromarray(img) # convert to numpy array
        draw = ImageDraw.Draw(img)
        draw_box(draw, *bbox, 0, 0, 'red', 2)
        img = np.array(img) # convert to numpy array
        
    return img, is_checked, bbox, bbox_norm

if __name__=='__main__':
    image_shape = (200, 200, 3)
    list_fonts = find_all_font_names()

    for _ in range(10):
        image, is_checked, bbox, bbox_norm = create_example(shape=image_shape, list_font_filepaths=list_fonts['filepath'].tolist(), draw_bbox=True)
    #     image, is_checked, bbox, bbox_norm = create_example(shape=image_shape, draw_bbox=False, line_color_threshold=1)
        bbox2 = (bbox_norm[0]*image_shape[1], bbox_norm[1]*image_shape[0], bbox_norm[2]*image_shape[1], bbox_norm[3]*image_shape[0])
        
        plt.figure(figsize=(5, 5))
        plt.title(f'checked; bbox = {bbox} {bbox2}' if is_checked else f'unchecked; bbox = {bbox} {bbox2}')
        plt.imshow(image)
        plt.xticks([])
        plt.yticks([])
        plt.show()